# SEA-TauBench Figure Plots

## Configuration

Change paths, filter settings, metric selections, labels, and export formats here when reusing the notebook with updated data.


In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path("tau3_result_plot")
CSV_PATH = PROJECT_DIR / "result_consolidate.csv"
FIG_DIR = PROJECT_DIR / "figures"

CACHE_DIR = PROJECT_DIR / ".cache"
MPL_CONFIG_DIR = CACHE_DIR / "matplotlib"
for directory in (FIG_DIR, CACHE_DIR, MPL_CONFIG_DIR, CACHE_DIR / "fontconfig"):
    directory.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(MPL_CONFIG_DIR))
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR))

EXPORT_FORMATS = ("pdf", "png", "svg")
EXPORT_DPI = 400

FILTER_SETTING = {
    "senario": [
        "1: english only",
        "2: multilingual tools",
        "3: crosslingual",
        "4: translated",
    ],
    "domain": ["airline", "retail", "telecom"],
    "normalized language/senario": [
        "english",
        "chinese",
        "indonesian",
        "thai",
        "vietnamese",
        "filipino",
        "tool_mix_2",
        "tool_mix_3",
        "tool_mix_4",
        "tool_mix_5",
    ],
    "normalized agent model": [
        "gpt-5-mini",
        "qwen3-235b-a22b-inst",
        "qwen3.6-35b-a3b",
        "kimi-k2.5",
    ],
}

PRIMARY_METRICS = ["pass@1", "pass@2", "pass@3"]
PAPER_METRICS = ["pass@1", "pass@3"]
FIGURE5_METRICS = ["pass@1", "rho@3"]
FIGURE6_METRICS = ["pass@1", "rho@3"]
ACTION_METRICS = ["read_actions", "write_actions", "db_match", "language_correctness"]

METRIC_RENAMES = {
    "pass^1": "pass@1",
    "pass^2": "pass@2",
    "pass^3": "pass@3",
    "rho^3": "rho@3",
    "Read Actions": "read_actions",
    "Write Actions": "write_actions",
    "DB Match": "db_match",
    "language_correctness": "language_correctness",
}

SCENARIO_LABELS = {
    1: "EN Baseline",
    2: "L2 Tools",
    3: "L2 Interaction",
    4: "L2 Domain",
}
SCENARIO_ORDER = [1, 3, 2, 4]
NON_BASELINE_SCENARIO_ORDER = [3, 2, 4]

LANGUAGE_ORDER = [
    "english",
    "thai",
    "vietnamese",
    "filipino",
    "indonesian",
    "chinese",
]
TOOL_MIX_ORDER = ["tool_mix_2", "tool_mix_3", "tool_mix_4", "tool_mix_5"]
LANGUAGE_LABELS = {
    "english": "EN",
    "chinese": "ZH",
    "indonesian": "ID",
    "thai": "TH",
    "vietnamese": "VI",
    "filipino": "FL",
    "tool_mix_2": "Mix 2",
    "tool_mix_3": "Mix 3",
    "tool_mix_4": "Mix 4",
    "tool_mix_5": "Mix 5",
}
LANGUAGE_DISPLAY_NAMES = {
    "english": "English",
    "chinese": "Chinese",
    "indonesian": "Indonesian",
    "thai": "Thai",
    "vietnamese": "Vietnamese",
    "filipino": "Filipino",
    "tool_mix_2": "Tool Mix 2",
    "tool_mix_3": "Tool Mix 3",
    "tool_mix_4": "Tool Mix 4",
    "tool_mix_5": "Tool Mix 5",
}

MODEL_ORDER = FILTER_SETTING["normalized agent model"]
PLOT_MODEL_KEYS = ["gpt-5-mini", "qwen3-235b-a22b-inst"]
FIG6_ALL_MODEL_SCENARIO_IDS = {3, 4}
MODEL_LABELS = {
    "gpt-5-mini": "GPT-5 Mini",
    "kimi-k2.5": "Kimi K2.5",
    "qwen3-235b-a22b-inst": "Qwen3-235B-A22B-INST",
    "qwen3.6-35b-a3b": "Qwen3.6-35B-A3B",
}

print(f"CSV_PATH: {CSV_PATH}")
print(f"FIG_DIR: {FIG_DIR}")
print("FILTER_SETTING:")
for key, values in FILTER_SETTING.items():
    print(f"  {key}: {values}")
print(f"Default plot model subset: {PLOT_MODEL_KEYS}")
print(f"Figure 6 all-model scenario IDs: {sorted(FIG6_ALL_MODEL_SCENARIO_IDS)}")


## Imports and Plot Style

The style uses colorblind-safe colors, vector-friendly font settings, compact paper proportions, and matplotlib-only plotting so the notebook runs in this local environment.


In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

OKABE_ITO = {
    "orange": "#E69F00",
    "sky_blue": "#56B4E9",
    "bluish_green": "#009E73",
    "yellow": "#F0E442",
    "blue": "#0072B2",
    "vermillion": "#D55E00",
    "reddish_purple": "#CC79A7",
    "black": "#000000",
}

LANGUAGE_PALETTE = {
    "EN": OKABE_ITO["orange"],
    "ZH": OKABE_ITO["sky_blue"],
    "ID": OKABE_ITO["blue"],
    "TH": OKABE_ITO["bluish_green"],
    "VI": OKABE_ITO["reddish_purple"],
    "FL": OKABE_ITO["vermillion"],
}

METRIC_PALETTE = {
    "pass@1": OKABE_ITO["orange"],
    "pass@2": OKABE_ITO["sky_blue"],
    "pass@3": OKABE_ITO["reddish_purple"],
    "rho@3": OKABE_ITO["reddish_purple"],
}

MODEL_PALETTE = dict(zip(MODEL_ORDER, [
    OKABE_ITO["orange"],
    OKABE_ITO["sky_blue"],
    OKABE_ITO["bluish_green"],
    OKABE_ITO["reddish_purple"],
]))

plt.style.use("default")
plt.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": EXPORT_DPI,
        "savefig.bbox": "tight",
        "savefig.facecolor": "white",
        "font.family": "DejaVu Sans",
        "font.size": 8,
        "axes.titlesize": 9,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "axes.linewidth": 0.7,
        "grid.linewidth": 0.45,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    }
)


def despine(ax):
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)


def save_publication_figure(fig, name, formats=EXPORT_FORMATS):
    saved_paths = []
    for ext in formats:
        out_path = FIG_DIR / f"{name}.{ext}"
        fig.savefig(out_path, dpi=EXPORT_DPI, bbox_inches="tight", facecolor="white")
        saved_paths.append(out_path)
    print("Saved:")
    for path in saved_paths:
        print(f"  {path}")
    return saved_paths


## Load, Filter, Normalize, and Deduplicate

Data rules:

- Keep rows that match `FILTER_SETTING` exactly.
- Use `normalized agent model` and `normalized language/senario` as the only model and language/scenario sources.
- Keep rows that have at least one primary pass metric.
- Deduplicate by scenario/domain/normalized language-scenario/normalized model, preferring `Finalized=True`, then the latest CSV row.


In [ ]:
def normalize_key_series(series):
    return series.astype("string").str.strip().str.lower()


def coerce_finalized(value):
    if pd.isna(value):
        return False
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def parse_scenario_id(series):
    return pd.to_numeric(
        series.astype("string").str.extract(r"^\s*(\d+)")[0],
        errors="coerce",
    ).astype("Int64")


def load_and_prepare(csv_path=CSV_PATH):
    raw = pd.read_csv(csv_path)
    raw.columns = [column.strip() for column in raw.columns]
    df = raw.reset_index(names="source_row").copy()

    for original, renamed in METRIC_RENAMES.items():
        if original in df.columns:
            df[renamed] = pd.to_numeric(df[original], errors="coerce")
        else:
            df[renamed] = np.nan

    df["scenario_raw"] = df["Senario"].astype("string").str.strip()
    df["scenario_id"] = parse_scenario_id(df["scenario_raw"])
    df["scenario_label"] = df["scenario_id"].map(SCENARIO_LABELS).fillna(df["scenario_raw"])

    df["domain_key"] = normalize_key_series(df["domain"])
    df["domain_label"] = df["domain_key"].str.title()
    df["language_key"] = normalize_key_series(df["normalized language/senario"])
    df["language_label"] = df["language_key"].map(LANGUAGE_LABELS).fillna(df["language_key"])
    df["language_display"] = df["language_key"].map(LANGUAGE_DISPLAY_NAMES).fillna(df["language_key"])
    df["language_group"] = np.where(df["language_key"].str.startswith("tool_mix_", na=False), "tool_mix", "language")
    df["tool_mix_n"] = pd.to_numeric(
        df["language_key"].str.extract(r"tool_mix_(\d+)")[0],
        errors="coerce",
    )

    df["model_key"] = normalize_key_series(df["normalized agent model"])
    df["model_label"] = df["model_key"].map(MODEL_LABELS).fillna(df["model_key"])
    df["finalized_bool"] = df["Finalized"].map(coerce_finalized)

    has_primary_metric = df[PRIMARY_METRICS].notna().any(axis=1)
    filter_mask = (
        df["scenario_raw"].isin(FILTER_SETTING["senario"])
        & df["domain_key"].isin(FILTER_SETTING["domain"])
        & df["language_key"].isin(FILTER_SETTING["normalized language/senario"])
        & df["model_key"].isin(FILTER_SETTING["normalized agent model"])
    )
    with_metrics = df.loc[filter_mask & has_primary_metric].copy()

    dedupe_keys = ["scenario_raw", "domain_key", "language_key", "model_key"]
    duplicate_summary = (
        with_metrics.groupby(dedupe_keys, dropna=False)
        .size()
        .reset_index(name="row_count")
        .query("row_count > 1")
        .sort_values("row_count", ascending=False)
    )
    clean = (
        with_metrics.sort_values(["finalized_bool", "source_row"])
        .drop_duplicates(dedupe_keys, keep="last")
        .sort_values("source_row")
        .reset_index(drop=True)
    )

    audit = {
        "raw_rows": len(raw),
        "rows_matching_filter": int(filter_mask.sum()),
        "filtered_rows_with_primary_metrics_before_dedupe": len(with_metrics),
        "clean_rows_after_dedupe": len(clean),
        "duplicate_groups_resolved": len(duplicate_summary),
        "excluded_rows_by_filter_or_missing_metrics": len(df) - len(with_metrics),
    }
    return raw, df, clean, duplicate_summary, audit


raw_df, normalized_df, clean_df, duplicate_summary, audit = load_and_prepare()
plot_df = clean_df.loc[clean_df["model_key"].isin(PLOT_MODEL_KEYS)].copy()

print("Audit")
for key, value in audit.items():
    print(f"  {key}: {value}")

print("\nClean scenarios:")
print(clean_df.groupby(["scenario_id", "scenario_label"]).size().to_string())

print("\nClean domains:")
print(clean_df.groupby("domain_key").size().to_string())

print("\nClean models available for Figure 6:")
print(clean_df.groupby("model_key").size().reindex(MODEL_ORDER).dropna().astype(int).to_string())

print("\nDefault plot model subset:")
print(plot_df.groupby("model_key").size().reindex(PLOT_MODEL_KEYS).dropna().astype(int).to_string())

print("\nDuplicate groups resolved:")
if duplicate_summary.empty:
    print("  None")
else:
    print(duplicate_summary.to_string(index=False))

preview_cols = [
    "scenario_label",
    "domain_label",
    "language_label",
    "model_label",
    "pass@1",
    "pass@2",
    "pass@3",
    "rho@3",
    "finalized_bool",
]
try:
    display(clean_df[preview_cols].head(12))
except NameError:
    print(clean_df[preview_cols].head(12).to_string(index=False))


## Figure 6: Model and Language Breakdown

Rows are metrics and columns are available non-baseline settings. Missing combinations are skipped without failing.


In [ ]:
def ordered_model_labels(frame):
    available = set(frame["model_key"].dropna())
    return [model for model in MODEL_ORDER if model in available]


MODEL_KEY_ORDER = ordered_model_labels(clean_df)
MODEL_LABEL_ORDER = [MODEL_LABELS.get(model, model) for model in MODEL_KEY_ORDER]
scenario_metric_availability = (
    clean_df.loc[clean_df["language_group"].eq("language")]
    .groupby("scenario_id")[PAPER_METRICS]
    .agg(lambda values: values.notna().any())
)
NON_BASELINE_SCENARIOS = [
    scenario
    for scenario in NON_BASELINE_SCENARIO_ORDER
    if scenario in scenario_metric_availability.index
    and bool(scenario_metric_availability.loc[scenario].any())
]
LANGUAGE_HUE_ORDER = [
    LANGUAGE_LABELS[language]
    for language in LANGUAGE_ORDER
    if language in set(clean_df["language_key"])
]
BASELINE_MODEL_KEYS = set(
    clean_df.loc[
        clean_df["scenario_id"].eq(1)
        & clean_df["language_key"].eq("english")
        & clean_df["language_group"].eq("language")
        & clean_df[FIGURE6_METRICS].notna().any(axis=1),
        "model_key",
    ]
)
FIG6_MODEL_ORDER_BY_SCENARIO = {}
FIG6_SCENARIO_SLOT_COUNTS = []
for scenario in NON_BASELINE_SCENARIOS:
    scenario_model_keys = set(
        clean_df.loc[
            clean_df["scenario_id"].eq(scenario)
            & clean_df["language_group"].eq("language")
            & clean_df[FIGURE6_METRICS].notna().any(axis=1),
            "model_key",
        ]
    )
    allowed_models = MODEL_KEY_ORDER if scenario in FIG6_ALL_MODEL_SCENARIO_IDS else [
        model for model in MODEL_KEY_ORDER if model in PLOT_MODEL_KEYS
    ]
    scenario_models = [
        model
        for model in allowed_models
        if model in scenario_model_keys or model in BASELINE_MODEL_KEYS
    ]
    FIG6_MODEL_ORDER_BY_SCENARIO[scenario] = scenario_models
    FIG6_SCENARIO_SLOT_COUNTS.append(max(1, len(scenario_models)))

fig, axes = plt.subplots(
    len(FIGURE6_METRICS),
    len(NON_BASELINE_SCENARIOS),
    figsize=(0.8 * sum(FIG6_SCENARIO_SLOT_COUNTS), 2.6875 * len(FIGURE6_METRICS)),
    sharey=True,
    squeeze=False,
    gridspec_kw={"width_ratios": FIG6_SCENARIO_SLOT_COUNTS},
)
fixed_bar_width = 0.12
hue_offsets = (np.arange(len(LANGUAGE_HUE_ORDER)) - (len(LANGUAGE_HUE_ORDER) - 1) / 2) * fixed_bar_width

FIGURE6_METRIC_LABELS = {"pass@1": "pass@1", "rho@3": r"$\rho^3$"}

for row_idx, metric in enumerate(FIGURE6_METRICS):
    for col_idx, scenario_id in enumerate(NON_BASELINE_SCENARIOS):
        ax = axes[row_idx, col_idx]
        scenario_subset = clean_df.loc[
            clean_df["scenario_id"].eq(scenario_id)
            & clean_df["language_group"].eq("language")
            & clean_df[metric].notna()
        ].copy()

        order = FIG6_MODEL_ORDER_BY_SCENARIO[scenario_id]
        english_baseline_subset = clean_df.loc[
            clean_df["scenario_id"].eq(1)
            & clean_df["language_key"].eq("english")
            & clean_df["model_key"].isin(order)
            & clean_df[metric].notna()
        ].copy()
        subset = pd.concat([english_baseline_subset, scenario_subset], ignore_index=True)
        if subset.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            ax.set_axis_off()
            continue

        x_positions = np.arange(len(order))
        summary = (
            subset.groupby(["model_key", "language_label"])[metric]
            .agg(["mean"])
            .reset_index()
        )

        for hue_idx, language_label in enumerate(LANGUAGE_HUE_ORDER):
            hue_summary = (
                summary.loc[summary["language_label"].eq(language_label)]
                .set_index("model_key")
                .reindex(order)
            )
            y_values = hue_summary["mean"].to_numpy(dtype=float).copy()
            valid = ~np.isnan(y_values)
            if not valid.any():
                continue
            bar_x = x_positions + hue_offsets[hue_idx]
            ax.bar(
                bar_x[valid],
                y_values[valid],
                width=fixed_bar_width * 0.9,
                color=LANGUAGE_PALETTE[language_label],
                edgecolor="white",
                linewidth=0.3,
            )

        ax.set_title(SCENARIO_LABELS.get(scenario_id, str(scenario_id)))
        ax.set_xlabel("")
        ax.set_ylabel(FIGURE6_METRIC_LABELS.get(metric, metric) if col_idx == 0 else "")
        ax.set_ylim(0, 1.02)
        ax.set_xticks(x_positions)
        ax.set_xticklabels([MODEL_LABELS.get(model, model) for model in order])
        ax.set_xlim(-0.55, max(0.55, len(order) - 0.45))
        ax.tick_params(axis="x", rotation=32)
        for label in ax.get_xticklabels():
            label.set_horizontalalignment("right")
        ax.grid(axis="y", alpha=0.35)
        despine(ax)

legend_handles = [
    Patch(facecolor=LANGUAGE_PALETTE[label], edgecolor="none")
    for label in LANGUAGE_HUE_ORDER
]
fig.legend(
    legend_handles,
    LANGUAGE_HUE_ORDER,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.015),
    ncol=min(6, len(LANGUAGE_HUE_ORDER)),
    frameon=False,
)
fig.tight_layout(rect=[0, 0.07, 1, 1])
save_publication_figure(fig, "figure6_model_language_breakdown")
plt.show()


## Figure 7: Degradation by Language with Model Dots

Each panel follows Figure 5 and overlays model-level averages as dots. Dots use the same color as the corresponding metric line.


In [ ]:
def available_language_order(frame):
    available = set(frame.loc[frame["language_group"].eq("language"), "language_key"].dropna())
    ordered = [language for language in LANGUAGE_ORDER if language in available]
    extras = sorted(available.difference(ordered))
    return ordered + extras


TARGET_LANGUAGES = [
    language
    for language in available_language_order(plot_df)
    if language != "english"
]
AVAILABLE_SCENARIO_IDS = set(plot_df["scenario_id"].dropna().astype(int))
FIGURE5_SCENARIOS = [scenario for scenario in NON_BASELINE_SCENARIO_ORDER if scenario in AVAILABLE_SCENARIO_IDS]
CONDITION_ORDER = ["EN Baseline"] + [SCENARIO_LABELS[scenario] for scenario in FIGURE5_SCENARIOS]
CONDITION_SHORT_LABELS = {
    "EN Baseline": "EN Baseline",
    "L2 Tools": "L2 Tools",
    "L2 Interaction": "L2 Interaction",
    "L2 Domain": "L2 Domain",
}


def build_figure5_frame(frame):
    baseline = frame.loc[
        frame["scenario_id"].eq(1)
        & frame["language_key"].eq("english")
        & frame["language_group"].eq("language")
    ].copy()
    baseline["condition"] = "EN Baseline"

    expanded_baseline = []
    for language in TARGET_LANGUAGES:
        copy = baseline.copy()
        copy["panel_language"] = language
        copy["panel_label"] = LANGUAGE_LABELS.get(language, language)
        expanded_baseline.append(copy)

    language_rows = frame.loc[
        frame["scenario_id"].isin(FIGURE5_SCENARIOS)
        & frame["language_key"].isin(TARGET_LANGUAGES)
        & frame["language_group"].eq("language")
    ].copy()
    language_rows["condition"] = language_rows["scenario_id"].map(SCENARIO_LABELS)
    language_rows["panel_language"] = language_rows["language_key"]
    language_rows["panel_label"] = language_rows["language_label"]

    parts = expanded_baseline + [language_rows]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


FIGURE7_BASE_METRIC_LABELS = {"pass@1": "pass@1", "rho@3": r"$\rho^3$"}
FIGURE7_BASE_METRIC_PALETTE = {"pass@1": METRIC_PALETTE["pass@1"], "rho@3": METRIC_PALETTE["rho@3"]}


In [ ]:
FIGURE7_METRICS = FIGURE5_METRICS
FIGURE7_METRIC_LABELS = FIGURE7_BASE_METRIC_LABELS
FIGURE7_METRIC_PALETTE = FIGURE7_BASE_METRIC_PALETTE
FIGURE7_ALL_MODEL_SCENARIO_IDS = {3, 4}


def figure7_allowed_models(condition):
    if condition == "EN Baseline":
        return MODEL_ORDER
    scenario_id = next(
        (key for key, value in SCENARIO_LABELS.items() if value == condition),
        None,
    )
    if scenario_id in FIGURE7_ALL_MODEL_SCENARIO_IDS:
        return MODEL_ORDER
    return PLOT_MODEL_KEYS


def build_figure7_frame(frame):
    figure7_frame = build_figure5_frame(frame).copy()
    allowed_parts = []
    for condition in CONDITION_ORDER:
        allowed_models = set(figure7_allowed_models(condition))
        allowed_parts.append(
            figure7_frame.loc[
                figure7_frame["condition"].eq(condition)
                & figure7_frame["model_key"].isin(allowed_models)
            ]
        )
    return pd.concat(allowed_parts, ignore_index=True) if allowed_parts else pd.DataFrame()


fig7_df = build_figure7_frame(clean_df)

n_panels = max(1, len(TARGET_LANGUAGES))
n_cols = min(3, n_panels)
n_rows = math.ceil(n_panels / n_cols)
fig = plt.figure(figsize=(2.55 * n_cols, 2.15 * n_rows))
grid = fig.add_gridspec(n_rows, n_cols * 2)
axes_flat = []
for panel_idx in range(len(TARGET_LANGUAGES)):
    row_idx = panel_idx // n_cols
    col_idx = panel_idx % n_cols
    panels_in_row = min(n_cols, len(TARGET_LANGUAGES) - row_idx * n_cols)
    row_offset = n_cols - panels_in_row
    start_col = col_idx * 2 + row_offset
    share_axis = axes_flat[0] if axes_flat else None
    axes_flat.append(fig.add_subplot(grid[row_idx, start_col : start_col + 2], sharey=share_axis))

x_positions = np.arange(len(CONDITION_ORDER))
score_ticks = np.linspace(0, 1, 6)
score_tick_labels = [f"{tick:g}" for tick in score_ticks]
metric_offsets = {
    metric: offset
    for metric, offset in zip(FIGURE7_METRICS, np.linspace(-0.045, 0.045, len(FIGURE7_METRICS)))
}
model_offsets = {
    model: offset
    for model, offset in zip(MODEL_ORDER, np.linspace(-0.03, 0.03, len(MODEL_ORDER)))
}

for ax, language in zip(axes_flat, TARGET_LANGUAGES):
    subset = fig7_df.loc[fig7_df["panel_language"].eq(language)]
    for metric in FIGURE7_METRICS:
        stats = (
            subset.groupby("condition")[metric]
            .agg(["mean", "std", "count"])
            .reindex(CONDITION_ORDER)
        )
        mean = stats["mean"]
        sem = stats["std"].fillna(0) / np.sqrt(stats["count"].clip(lower=1))
        y_values = mean.to_numpy(dtype=float).copy()
        y_errors = sem.to_numpy(dtype=float).copy()
        y_errors[np.isnan(y_values)] = np.nan

        dot_summary = (
            subset.groupby(["condition", "model_key"])[metric]
            .mean()
            .reset_index()
        )
        for condition_idx, condition in enumerate(CONDITION_ORDER):
            allowed_models = figure7_allowed_models(condition)
            condition_dots = (
                dot_summary.loc[dot_summary["condition"].eq(condition)]
                .set_index("model_key")
                .reindex(allowed_models)
                .dropna(subset=[metric])
                .reset_index()
            )
            if condition_dots.empty:
                continue
            dot_x = np.array(
                [
                    x_positions[condition_idx]
                    + metric_offsets[metric]
                    + model_offsets.get(model, 0)
                    for model in condition_dots["model_key"]
                ],
                dtype=float,
            )
            ax.scatter(
                dot_x,
                condition_dots[metric].to_numpy(dtype=float),
                s=18,
                color=FIGURE7_METRIC_PALETTE[metric],
                edgecolor="white",
                linewidth=0.35,
                alpha=0.72,
                zorder=4,
            )
        ax.errorbar(
            x_positions,
            y_values,
            yerr=y_errors,
            marker="o",
            markersize=4,
            linewidth=1.6,
            capsize=2.5,
            color=FIGURE7_METRIC_PALETTE[metric],
            label=FIGURE7_METRIC_LABELS[metric],
            zorder=3,
        )

    ax.set_title(f"{LANGUAGE_LABELS.get(language, language)} ({LANGUAGE_DISPLAY_NAMES.get(language, language)})")
    ax.set_xticks(x_positions)
    ax.set_xticklabels([CONDITION_SHORT_LABELS[label] for label in CONDITION_ORDER], rotation=20, ha="right")
    ax.set_ylim(0, 1.02)
    ax.set_yticks(score_ticks)
    ax.set_yticklabels(score_tick_labels)
    ax.tick_params(axis="y", labelleft=True)
    ax.set_ylabel("Score")
    ax.grid(axis="y", alpha=0.35)
    despine(ax)

handles, labels = axes_flat[0].get_legend_handles_labels()
if handles:
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.015),
        ncol=len(labels),
        frameon=False,
    )
fig.tight_layout(rect=[0, 0.10, 1, 1])
save_publication_figure(fig, "figure7_degradation_by_language_with_model_dots")
plt.show()


## Export Manifest

Run this cell after plotting to list this notebook's generated artifacts.


In [ ]:
FIGURE_EXPORT_STEMS = [
    "figure6_model_language_breakdown",
    "figure7_degradation_by_language_with_model_dots",
]
exported_files = sorted(
    path
    for path in FIG_DIR.iterdir()
    if path.stem in FIGURE_EXPORT_STEMS and path.suffix.lower() in {".pdf", ".png", ".svg"}
)
print(f"Exported {len(exported_files)} files to {FIG_DIR}")
for path in exported_files:
    print(path)
